# ===============================================================
# Phase 2: Clean & Chunk
#  - Input : data/raw/jenosize_articles_sorted.jsonl
#  - Output: data/processed/articles_clean.jsonl, data/processed/chunks.jsonl
# ===============================================================

In [ ]:

from pathlib import Path
from dotenv import load_dotenv
import os, json, re

load_dotenv()

PROJECT_ROOT = Path(r"D:\mini-jane-demo")

RAW_PATH       = PROJECT_ROOT / "data" / "raw"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"

RAW_PATH.mkdir(parents=True, exist_ok=True)
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

RAW_SORTED   = RAW_PATH / "jenosize_articles_sorted.jsonl"
ARTICLES_OUT = PROCESSED_PATH / "articles_clean.jsonl"
CHUNKS_OUT   = PROCESSED_PATH / "chunks.jsonl"

print("✅ PROJECT_ROOT :", PROJECT_ROOT)
print("✅ RAW_SORTED   :", RAW_SORTED, "exists?", RAW_SORTED.exists())
print("✅ ARTICLES_OUT :", ARTICLES_OUT)
print("✅ CHUNKS_OUT   :", CHUNKS_OUT)


✅ PROJECT_ROOT : D:\mini-jane-demo
✅ RAW_SORTED   : D:\mini-jane-demo\data\raw\jenosize_articles_sorted.jsonl exists? True
✅ ARTICLES_OUT : D:\mini-jane-demo\data\processed\articles_clean.jsonl
✅ CHUNKS_OUT   : D:\mini-jane-demo\data\processed\chunks.jsonl


In [14]:
# ---------- Cleaning helpers (pure functions) ----------
def clean_tail(text: str) -> str:
    """ลบ 'Loading…', เว้นวรรค/บรรทัดว่างซ้ำ"""
    t = (text or "").strip()
    t = re.sub(r"\bLoading…?$", "", t, flags=re.IGNORECASE).strip()
    t = re.sub(r"[ \t]+\n", "\n", t)        # space ท้ายบรรทัด
    t = re.sub(r"\n{3,}", "\n\n", t)        # บรรทัดว่างซ้อน
    return t

def is_short(text: str, min_words=120) -> bool:
    """ตัดบทความที่สั้นเกินไป (กัน noise)"""
    return len((text or "").split()) < min_words

def normalize_record(rec: dict) -> dict:
    """เติม field สำคัญให้ครบ + คำนวณ word_count"""
    rec["title"]      = rec.get("title") or "Untitled"
    rec["url"]        = rec.get("url")
    rec["category"]   = rec.get("category") or "Unknown"
    rec["language"]   = "en"   # ชุดนี้เป็นบทความภาษาอังกฤษ
    rec["word_count"] = len((rec.get("text") or "").split())
    return rec


In [16]:
# ---------- Clean & write out articles ----------
assert RAW_SORTED.exists(), f"Not found: {RAW_SORTED}"
ARTICLES_OUT.write_text("", encoding="utf-8")  # reset ให้ไฟล์ใหม่เสมอ

kept, dropped = 0, 0
with open(RAW_SORTED, "r", encoding="utf-8") as fin, \
     open(ARTICLES_OUT, "a", encoding="utf-8") as fout:
    for line in fin:
        if not line.strip():
            continue
        rec = json.loads(line)
        text = clean_tail(rec.get("text",""))
        if is_short(text, min_words=120):
            dropped += 1
            continue
        rec["text"] = text
        norm = normalize_record(rec)
        json.dump(norm, fout, ensure_ascii=False); fout.write("\n")
        kept += 1

print(f"✅ Cleaned articles -> {ARTICLES_OUT} | kept={kept}, dropped={dropped}")


✅ Cleaned articles -> D:\mini-jane-demo\data\processed\articles_clean.jsonl | kept=123, dropped=0


In [18]:
# ---------- Chunking helpers ----------
def approx_tokens(text: str) -> int:
    """ประมาณจำนวนโทเค็น: คำ/0.75 (หยาบแต่พอใช้งานได้)"""
    return max(1, int(len(text.split()) / 0.75))

def split_sections(text: str):
    """
    ถ้ามี Markdown หัวข้อ (##, ###) จะแยกตามหัวข้อ
    ถ้าไม่มี ให้รวมเป็น section เดียวชื่อ 'Body'
    """
    has_md_headers = bool(re.search(r"(?m)^\s*#{2,3}\s+", text))
    if has_md_headers:
        blocks = []
        current_header = "Intro"
        current_buf = []
        for line in text.splitlines():
            if re.match(r"^\s*#{2,3}\s+", line):
                if current_buf:
                    blocks.append({"section": current_header, "text": "\n".join(current_buf).strip()})
                    current_buf = []
                current_header = re.sub(r"^\s*#{2,3}\s+", "", line).strip()
            else:
                current_buf.append(line)
        if current_buf:
            blocks.append({"section": current_header, "text": "\n".join(current_buf).strip()})
        return [b for b in blocks if b["text"]]
    else:
        paras = [p.strip() for p in re.split(r"\n{2,}", text) if p.strip()]
        return [{"section":"Body", "text":"\n\n".join(paras)}]

def rechunk(section_text: str, target_tokens=350, overlap_tokens=100):
    """
    ตัดข้อความเป็นก้อน ๆ ใกล้ target_tokens พร้อม overlap ถอยหลัง ~100 โทเค็น
    """
    sents = re.split(r"(?<=[\.!?])\s+|\n{2,}", section_text.strip())
    chunks, cur, cur_tok = [], [], 0
    for s in sents:
        if not s.strip():
            continue
        t = approx_tokens(s)
        if cur_tok + t <= target_tokens or not cur:
            cur.append(s); cur_tok += t
        else:
            chunks.append(" ".join(cur).strip())
            # ทำ overlap โดยถอยหลังสะสมจนถึง overlap_tokens
            ov, ov_tok = [], 0
            for ss in reversed(cur):
                ov_tok += approx_tokens(ss)
                ov.insert(0, ss)
                if ov_tok >= overlap_tokens:
                    break
            cur = ov + [s]
            cur_tok = sum(approx_tokens(x) for x in cur)
    if cur:
        chunks.append(" ".join(cur).strip())
    # กรองชิ้นสั้นผิดปกติ
    return [c for c in chunks if len(c.split()) >= 40]


In [19]:
# ---------- Build chunks file ----------
CHUNKS_OUT.write_text("", encoding="utf-8")

total_articles, total_chunks = 0, 0
with open(ARTICLES_OUT, "r", encoding="utf-8") as fin, \
     open(CHUNKS_OUT, "a", encoding="utf-8") as fout:
    for line in fin:
        rec = json.loads(line)
        sections = split_sections(rec["text"])
        idx = 0
        for sec in sections:
            parts = rechunk(sec["text"], target_tokens=350, overlap_tokens=100)
            for p in parts:
                idx += 1; total_chunks += 1
                out = {
                    "doc_id": rec["url"],
                    "url": rec["url"],
                    "title": rec["title"],
                    "category": rec["category"],
                    "section": sec["section"],
                    "chunk_index": idx,
                    "text": p,
                    "word_count": len(p.split()),
                    "language": rec["language"]
                }
                json.dump(out, fout, ensure_ascii=False); fout.write("\n")
        total_articles += 1

print(f"✅ Chunked: {total_articles} articles → {total_chunks} chunks")
print(f"📄 Saved: {CHUNKS_OUT}")


✅ Chunked: 123 articles → 616 chunks
📄 Saved: D:\mini-jane-demo\data\processed\chunks.jsonl


In [20]:
# ---------- Quick sanity checks ----------
import statistics as stats

words = []
cats = {}
with open(CHUNKS_OUT, "r", encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        wc = r["word_count"]
        words.append(wc)
        cats[r["category"]] = cats.get(r["category"], 0) + 1

print("Total chunks:", len(words))
if words:
    p50 = int(stats.median(words))
    p90 = int(sorted(words)[int(0.9*len(words))-1])
    print("Word count p50/p90/max:", p50, p90, max(words))
print("By category:", cats)


Total chunks: 616
Word count p50/p90/max: 254 265 376
By category: {'Experience the New World': 61, 'Futurist': 86, 'Real-time Marketing': 96, 'Transformation & Technology': 191, 'Understand People & Consumer': 94, 'Utility for Our World': 88}
